# Curation of Crop Data

## 1. Plan

- Configure Environment.
- Retrieve weather data from URL.
  - Raw data, with unnecessary metadata removed.
- Download and save as csv for offline use.
- Parse csv into dataframe for wrangling in readiness for analytics.
- Conduct exploratory data analysis to identify interesting features for exploration.
- Analyse interesting feature interactions with statistical models/ML.
- Explore interesting interactions with further EDA and ML.

### Iterate All Files

- New dirs for csv files
  - Raw csv from url
  - Yields csv from pandas
- Load Yields csv into SQLite

## 2. Environment Setup and Configuration

### Test kernel

Just for checking the environment is working.

In [ ]:
txt = "Hello"
print(txt)

### Import Required Libraries

In [ ]:
# For calling data repository URL
import requests
import io

# for file and directory manipulation
import os

# For manipulating data with Pandas
import pandas as pd

# using the sqlite3 database
import sqlite3

# Ensure the odfpy library is installed
%pip install odfpy

## 3. Extract Data from Government Online Repository 

- Source data from URL.
- Parse into Pandas DataFrame.
- List sheets in ODF workbook.
- For each sheet in ODF workbook:
  - Extract as dataframe.
  - Export to local storage as CSV.

### Region Labels

Need to consider how the data is going to be aggregated. This needs to match with the aggregation of weather data, so will have to aggregate to the lowest common denominator.

Initial EDA may hihglight regional or country level variation highlighting features for exploration.

#### Crop Data Grouping Lists

These lists show how the data is presented in the source data.

|ID|Crop Regions|
|--|------------|
|1|United Kingdom|
|2|England|
|3|North East|
|4|North West and Merseyside|
|5|Yorkshire & The Humber|
|6|East Midlands|
|7|West Midlands|
|8|Eastern|
|9|South East and London|
|10|South West|
|11|Wales|
|12|Scotland|
|13|Northern Ireland|

### Source data from Government repository

Try following url: <https://assets.publishing.service.gov.uk/media/68e778c1e5f463a62cb98588/UK-cops-webseries-251009a.ods>

Objects:

- label: **buffer_object**, object type: **_io.BytesIO**

In [ ]:
# N.B. Need to ensure odfpy is installed in the environment. This allows the ODS binary to be parsed.

# Creates a ByteIO Class Object

url = "https://assets.publishing.service.gov.uk/media/68e778c1e5f463a62cb98588/UK-cops-webseries-251009a.ods"
try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()  # Raises for 4xx/5xx codes.
    buffer_object = io.BytesIO(response.content)
    print(f"Download succeeded: {len(response.content)} bytes")
except requests.exceptions.RequestException as e:
    buffer_object = None
    print(f"ERROR: Unable to download ODS file from {url}: {e}")
except Exception as e:
    buffer_object = None
    print(f"Unexpected error while creating buffer_object: {e}")


### Identify the sheets in the workbook

Objects:
- label: **list_odf_sheets**, object type: **list**
- label: **sheets_list**, object typeL **.csv**


In [ ]:
# https://pandas.pydata.org/docs/dev/reference/api/pandas.ExcelFile.sheet_names.html
# this outputs an ExcelFile Class Object

list_odf_sheets = []

if buffer_object is None:
    raise RuntimeError("buffer_object is not available; cannot read workbook")

try:
    crops_file = pd.ExcelFile(path_or_buffer=buffer_object, engine="odf")
    list_odf_sheets = list(crops_file.sheet_names)
    print(f"Sheets found: {len(list_odf_sheets)}")
except Exception as e:
    list_odf_sheets = []
    print(f"ERROR: Unable to read sheets from ODS workbook: {e}")

list_odf_sheets


### Output list to a .csv using Pandas

In [ ]:
# Ensure that the directory exists before writing files
os.makedirs(f'data/crops/', exist_ok=True)

df_odf_list = pd.DataFrame(list_odf_sheets)

df_odf_list.to_csv('data/crops/sheet_list.csv', index=False)

df_odf_list.head()

### Parse each sheet into separate .csv

For each sheet with suffix 'Region' parse into from **buffer_object** into dataframes and output as local .csv files.

Objects:
- label: **raw_wheat.csv**, object type: **.csv**
- label: **raw_winter_barley.csv**, object type: **.csv**
- label: **raw_spring_barley.csv**, object type: **.csv**
- label: **raw_total_barley.csv**, object type: **.csv**
- label: **raw_oats.csv**, object type: **.csv**
- label: **raw_osr.csv**, object type: **.csv**

Locally saving the file allows decoupling the need for network calls, allowing for faster I/O operations. This also allows me to continue development whilst internet connection was not availalble, e.g. whilst travelling on the train.

Exporting locally to .csv files does come at the cost of storage. This can be mitigated by deleting the files once processing is completed, retaining only the required data for reference. However, the size of data is very small and easily maintained on even the most limited of systems.

### Output into "raw" directory

In [ ]:
# Define the prefix
prefix = 'Regional'

# Ensure that the directory exists before writing files
os.makedirs('data/crops/raw', exist_ok=True)

if not list_odf_sheets:
    print("WARNING: No sheets detected. Skipping raw CSV export step.")

# Use for loop to list through list of sheet names, parse sheet into dataframe, and output locally as .csv file.
for name in list_odf_sheets:
    if name.startswith(prefix):
        try:
            df_sheet = pd.ExcelFile(path_or_buffer=buffer_object, engine="odf").parse(name)
            output_path = f'data/crops/raw/{name}.csv'
            df_sheet.to_csv(output_path, index=False)
            print(f"Exported {name} -> {output_path}")
        except Exception as e:
            print(f"ERROR: failed to export sheet {name}: {e}")


## 4. Clean data

- For each sheet CSV:
  - load into dataframe.
  - crop to new dataframe with relevant rows and columns.
  - Reset column headers and index to years.
- Save as csv in "yeilds" directory.

### 4.1. Process all files

#### List of Crops

In [ ]:
crop_file_names = [
			'Regional_oats',
			'Regional_OSR',
			'Regional_spring_barley',
			'Regional_total_barley',
			'Regional_wheat',
			'Regional_winter_barley'
]

In [ ]:
for crop in crop_file_names:
	print(f'crop name = {crop[9:]}')

#### Loop through and process all files to transformed csv

This is a consolidated process for apply this across all the crop files at once. More detailed run through follow showing the transformation of just the single spring barley csv file.

In [ ]:
# parameterise start and end rows for yield section.
yield_headers_start = 17
yield_rows = 13

for crop in crop_file_names:
    try:
        file_path = f'data/crops/raw/{crop}.csv'
        df_raw = pd.read_csv(file_path, header=yield_headers_start, nrows=yield_rows)

        # Drop unwanted columns.
        unwanted_columns = ["% change 2025/2024", "2025 confidence interval", "Unnamed: 30", "5 year average"]
        df_crop = df_raw.drop(columns=[col for col in unwanted_columns if col in df_raw.columns], errors='ignore')

        # Rename columns for consistency.
        df_crop.rename(columns={
            'Yields (1)': 'Year',
            '2022(1)': '2022',
            '2022(a)': '2022',
        }, inplace=True)

        # Transpose dataframe with regions as columns and years as rows.
        df_long = df_crop.transpose()

        # Make the first row, row index 0, the column names.
        if df_long.empty:
            raise ValueError(f"Extracted table empty for crop {crop}")

        df_long.columns = df_long.iloc[0]

        # Reset the dataframe to start without the first row that has now become the column names.
        df_long = df_long[1:].copy()

        # Clear the column heading name.
        df_long.columns.name = None

        # Reset the index to make years a regular column
        df_long.reset_index(inplace=True)

        # Rename new column for clarity (the index becomes a column named 'index' by default)
        df_long.rename(columns={'index': 'Year'}, inplace=True)

        # Convert data types
        for column in df_long.columns:
            if column == 'Year':
                df_long[column] = pd.to_numeric(df_long[column], errors='coerce').astype('Int64')
            else:
                df_long[column] = pd.to_numeric(df_long[column], errors='coerce').astype(float)

        # Save to "yields" directory as transformed csv file.
        os.makedirs('data/crops/yields/', exist_ok=True)

        name = f'yields_{crop[9:]}'
        df_long.to_csv(f'data/crops/yields/{name}.csv', index=False)
        print(f"Processed and saved {name}.csv")

    except FileNotFoundError:
        print(f"ERROR: raw file not found for crop {crop}: {file_path}")
    except Exception as e:
        print(f"ERROR processing crop {crop}: {e}")


### 4.2. Detailed Single Process

#### Load single crop .csv into DataFrame

Looking at the original odf download, we can determine the row in which the year column headers can be found. Since all crop sheets have identical structure, we can use the same logic.

- **header=17** skips to the row where the Yield data starts with the years as columns.
- **nrows=13** specifies the number of rows to retrieve, those with all the regions before the next dataset.

In [ ]:
# Will start with the Regional_spring_barley.csv
df_sb_raw = pd.read_csv('data/crops/raw/Regional_spring_barley.csv', header=17, nrows = 13) 

# This represents just the yield data required for this project
df_sb_raw.head()

#### Drop unwanted columns

In [ ]:
# Drop columns.
df_sb_crop = df_sb_raw.drop(["% change 2025/2024","2025 confidence interval","Unnamed: 30","5 year average"], axis=1)

# Rename columns for consistency.
df_sb_crop.rename(columns={
	'Yields (1)':'Year',
	'2022(1)':'2022',
	},
	inplace=True)
df_sb_crop.head()

#### Transpose data with years as rows

In [ ]:
# ref https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transpose.html

df_sb_long = df_sb_crop.transpose()

df_sb_long.head()

#### Reset column index to correct column index

In [ ]:
# Make the first row, row index 0, the column names.
df_sb_long.columns = df_sb_long.iloc[0]

df_sb_long.head(5)

In [ ]:
# Reset the dataframe to start without the first row that has now become the column names.
df_sb_long = df_sb_long[1:].copy()

df_sb_long.head()

In [ ]:
# Clear the column heading name.
df_sb_long.columns.name = None

df_sb_long.head(5)

#### Update the Year Index

The Years column is currently set as the index. To load this into the SQLite database as a column, this needs to be reset.

In [ ]:
# Reset the index to make years a regular column
df_sb_long.reset_index(inplace=True)

# Optionally rename the new column for clarity (the index becomes a column named 'index' by default)
df_sb_long.rename(columns={'index': 'Year'}, inplace=True)

df_sb_long.head(5)

#### Convert data types

In [ ]:
# Check types
df_sb_long.dtypes

In [ ]:
# For each column in the dataframe, convert the data type to float.
for column in df_sb_long.columns:
	if column == 'Year':
		df_sb_long[column] = df_sb_long[column].astype(int)

	elif column != 'Year':  # Skip the 'Year' column since it should remain as a string or integer
		df_sb_long[column] = df_sb_long[column].astype(float)

# Show new data types.
df_sb_long.dtypes

In [ ]:
df_sb_long

#### Save as csv in "yields" directory

In [ ]:
# Ensure that the directory exists before writing files
os.makedirs(f'data/crops/yields/', exist_ok=True)

name = "yields_spring_barley"

df_sb_long.to_csv(f'data/crops/yields/{name}.csv', index=False)

## 5. Load into database

### Single File Prototype

#### Prerequisites for SQLite database

In [ ]:
os.makedirs('data', exist_ok=True)
project_db  = 'data/project_db.db'

#### Load table "spring_barley"

In [ ]:
conn = sqlite3.connect(project_db)
table_name = 'spring_barley_yields'

df_sb_long.to_sql(
		name=table_name,
		con=conn,
		if_exists='replace',
		index=False
)

### Loop Load All

In [ ]:
# Get all file names to loop through.
yield_file_list: list = os.listdir('data/crops/yields')

print(yield_file_list)

In [ ]:
# Use file name as table name. Truncate to remove ".csv".
for yield_file in yield_file_list:
	table_name = yield_file[:-4]
	print(table_name)

In [ ]:
for yield_file in yield_file_list:

	# Parse file to dataframe.
	df_yield = pd.read_csv(f'data/crops/yields/{yield_file}')

	# Truncate file name for table name.
	table_name = yield_file[:-4]

	# open connection to database.
	conn = sqlite3.connect(project_db)

	try:
		# Load dataframe to sqlite.
		df_yield.to_sql(
			name=table_name,
			con=conn,
			if_exists='replace',
			index=False
		)

		print(f'Table {table_name} updated')

	except Exception as e:
		print(f"Error: {(str(e))}")
	
	finally:
		# Close connection
		conn.close()


### Sample SQLite Query

In [ ]:
conn = sqlite3.connect(project_db)

df_db_sp_yields = pd.read_sql('SELECT * FROM spring_barley_yields', conn)
conn.close()

df_db_sp_yields

In [ ]:
conn = sqlite3.connect(project_db)

df_db_sp_yields = pd.read_sql('SELECT "Year","North East" FROM spring_barley_yields', conn)
conn.close()

df_db_sp_yields